# SDXL Kaggle GPU backend

Ephemeral SDXL text-to-image backend for a Telegram bot, running on **free Kaggle GPU**.
The model checkpoint is **not** bundled — you supply it at runtime via a Kaggle secret.

## One-time setup (right-hand panel)
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On** (needed to download the model + open the tunnel).
3. **Add-ons → Secrets** — add these, then tick them ON for this notebook:
   - `WORKER_REGISTER_URL` = `https://<your-worker>.workers.dev/register-backend`
   - `REGISTRATION_SECRET` = the same secret set in Cloudflare
   - `TELEGRAM_BOT_TOKEN` = your bot token (so images are pushed to Telegram)
   - `MODEL_URL` = direct download link to your SDXL `.safetensors` checkpoint

## Run
Use **Run All**. Keep the tab open; the session stays alive up to ~9 hours.
When it prints `Registered temporary backend: https://...`, open the bot and send `/generate <prompt>`.


## 1) Install deps + cloudflared

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers", "transformers", "accelerate", "safetensors",
    "fastapi", "uvicorn", "python-multipart", "requests", "pillow",
    "nest-asyncio", "gradio"], check=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

import torch
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings > Accelerator > GPU)."
print("GPU:", torch.cuda.get_device_name(0))


## 2) Load config from Kaggle Secrets + fetch backend code

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WORKER_REGISTER_URL"] = secrets.get_secret("WORKER_REGISTER_URL").strip()
os.environ["REGISTRATION_SECRET"] = secrets.get_secret("REGISTRATION_SECRET").strip()
os.environ["MODEL_URL"] = secrets.get_secret("MODEL_URL").strip()
try:
    os.environ["TELEGRAM_BOT_TOKEN"] = secrets.get_secret("TELEGRAM_BOT_TOKEN").strip()
except Exception:
    print("WARNING: TELEGRAM_BOT_TOKEN secret not set/enabled - images will NOT be pushed to Telegram.")

# Pull the backend app (backend/app.py) from this repo so it is importable.
REPO = "https://github.com/blue-boy-questions/kaggle-sdxl-backend.git"
DEST = "/kaggle/working/kaggle-sdxl-backend"
if not os.path.isdir(DEST):
    subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=True)
sys.path.insert(0, DEST)
os.environ.setdefault("OUTPUT_DIR", "/kaggle/working/sdxl_outputs")
print("Config loaded. Register URL:", os.environ["WORKER_REGISTER_URL"])


## 3) Download the model + load pipeline (a few minutes)

In [ ]:
import os, requests, torch
from diffusers import StableDiffusionXLPipeline
from backend.app import set_pipeline

MODEL_URL = os.environ["MODEL_URL"]  # supplied via Kaggle secret, never hardcoded
MODEL_PATH = "/kaggle/working/model.safetensors"

if not (os.path.isfile(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1_000_000_000):
    print("Downloading model ...")
    with requests.get(MODEL_URL, stream=True, allow_redirects=True, timeout=180) as r:
        r.raise_for_status()
        with open(MODEL_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    print("Downloaded:", round(os.path.getsize(MODEL_PATH)/1e9, 2), "GB")

pipe = StableDiffusionXLPipeline.from_single_file(
    MODEL_PATH, torch_dtype=torch.float16, use_safetensors=True,
)

# IP-Adapter forces every call to require a reference image; keep it OFF for
# pure text-to-image. Set True only if you wire a reference image through.
LOAD_IP_ADAPTER = False
if LOAD_IP_ADAPTER:
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                         weight_name="ip-adapter_sdxl.bin")

# Choose the starting device. GPU is fast but burns Kaggle GPU quota;
# CPU is free (no quota) but slow (minutes per image). You can also switch
# later at runtime from Telegram via /device without reloading.
import torch
START_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(START_DEVICE)
if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()
if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_slicing"):
    pipe.vae.enable_slicing()

set_pipeline(pipe, START_DEVICE)
print("Model loaded on", START_DEVICE.upper())


## 4) Serve the API + Gradio, open the tunnel, register with the Worker

In [ ]:
import os, re, subprocess, threading, time
import nest_asyncio, requests, uvicorn
from backend.app import app, attach_gradio
import backend.app as backend_app

# Mount the Gradio web app at /app (same server) for the Telegram Mini App button.
attach_gradio("/app")

assert backend_app.PIPELINE is not None, "Run the model cell first (set_pipeline)."
nest_asyncio.apply()

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()

health = None
for _ in range(30):
    try:
        health = requests.get("http://127.0.0.1:8000/health", timeout=5).json()
        if health.get("ok"):
            break
    except Exception:
        pass
    time.sleep(1)
assert health and health.get("model_loaded"), f"Backend not healthy: {health}"
print("Local API healthy:", health)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2); continue
    print(line, end="")
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break
assert public_url, "Quick Tunnel URL was not found"

resp = requests.post(
    os.environ["WORKER_REGISTER_URL"].strip(),
    headers={"Authorization": "Bearer " + os.environ["REGISTRATION_SECRET"].strip(),
             "Content-Type": "application/json"},
    json={"url": public_url, "ttl_seconds": 32400, "device": backend_app.DEVICE},
    timeout=20,
)
resp.raise_for_status()
print("Registered temporary backend:", public_url)
print("Mini App (Gradio) URL:", public_url + "/app")
print(resp.json())


## 5) Keep-alive (optional)
Kaggle may idle-stop an interactive session. Run this cell LAST to hold the
kernel busy so the tunnel stays registered. Interrupt it when you're done.


In [ ]:
import time
print("Keeping the backend alive. Interrupt (stop) this cell to shut down.")
while True:
    time.sleep(300)
